# E2E Finance FAQ Non-Instruction Fine-Tuning (Colab)
This notebook sets up the repo on Colab, prepares data, builds token blocks, applies LoRA/QLoRA, and runs non-instruction fine-tuning.

In [ ]:
# 1) Runtime check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Set input corpus path (no Google Drive required)
from pathlib import Path

# Put your corpus at this exact path in Colab.
# Example accepted formats: CSV with complaint narratives.
INPUT_CORPUS_PATH = Path('/content/complaints-2026-07-04_02_07.csv')

# Optional: fallback auto-discovery pattern in /content.
INPUT_CORPUS_GLOB = 'complaints-*.csv'

print('INPUT_CORPUS_PATH:', INPUT_CORPUS_PATH)

In [ ]:
# 3) Clone or refresh repository
import os
REPO_URL = 'https://github.com/Swapperss/FinanceQAAssistance.git'
PROJECT_ROOT = '/content/FinanceQAAssistance'
if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print('Repository already exists, pulling latest changes...')
    %cd {PROJECT_ROOT}
    !git pull
%cd {PROJECT_ROOT}
!git branch --show-current

In [ ]:
# 4) Install dependencies
!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install -U transformers accelerate peft bitsandbytes datasets

In [ ]:
# 5) Load project config and logger
import sys
from pathlib import Path
project_root = Path('/content/FinanceQAAssistance').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from config.config import Config
from utils.logger import get_logger
config = Config()
logger = get_logger('non_instruction_colab')
logger.info('Colab E2E notebook started')
print('Loaded config model:', config.model_name)

In [ ]:
# 6) Create required folders
raw_dir = project_root / config.raw_data_dir
non_instruction_dir = project_root / config.non_instruction_dir
artifacts_output_dir = project_root / config.output_dir
artifacts_adapter_dir = project_root / config.adapter_dir
for p in [raw_dir, non_instruction_dir, artifacts_output_dir, artifacts_adapter_dir, project_root / 'logs']:
    p.mkdir(parents=True, exist_ok=True)
print('raw_dir:', raw_dir)
print('non_instruction_dir:', non_instruction_dir)
print('output_dir:', artifacts_output_dir)
print('adapter_dir:', artifacts_adapter_dir)

In [ ]:
# 7) Resolve input corpus and place it in project data/raw
import shutil
from pathlib import Path

target_csv_path = raw_dir / config.preferred_raw_csv_filename

if INPUT_CORPUS_PATH.exists():
    shutil.copy2(INPUT_CORPUS_PATH, target_csv_path)
    print('Copied input corpus to:', target_csv_path)
else:
    # Auto-discover likely corpus files in /content if INPUT_CORPUS_PATH is not set/found.
    matches = sorted(Path('/content').glob(INPUT_CORPUS_GLOB))
    if matches:
        source_path = matches[-1]
        shutil.copy2(source_path, target_csv_path)
        print('Auto-discovered and copied corpus:', source_path)
        print('Copied to:', target_csv_path)
    else:
        raise FileNotFoundError(
            'No input corpus found. Upload a CSV to /content/complaints-2026-07-04_02_07.csv '
            f'or place a file matching {INPUT_CORPUS_GLOB} under /content.'
        )

print('CSV exists in raw_dir:', target_csv_path.exists())

In [ ]:
# 8) Extract raw text from complaints CSV
import pandas as pd
csv_path = raw_dir / config.preferred_raw_csv_filename
if not csv_path.exists():
    matches = sorted(raw_dir.glob(config.raw_csv_pattern))
    if not matches:
        raise FileNotFoundError(f'No complaints CSV found in {raw_dir}')
    csv_path = matches[-1]
out_path = non_instruction_dir / config.raw_text_filename
df = pd.read_csv(csv_path, dtype=str, low_memory=False)
text_col = config.source_text_column
if text_col not in df.columns:
    raise ValueError(f'Column not found: {text_col}')
texts = df[text_col].dropna().astype(str).str.strip()
texts = texts[texts.str.len() > config.min_chars_per_paragraph]
out_path.write_text('\n\n'.join(texts.tolist()), encoding='utf-8')
logger.info('Raw extraction complete')
logger.info('Input rows: %s', len(df))
logger.info('Extracted text rows: %s', len(texts))
logger.info('Saved file: %s', out_path.resolve())
print('Extracted rows:', len(texts))
print('Saved to:', out_path.resolve())

In [ ]:
# 9) Clean text and create final non-instruction corpus
import re
in_path = non_instruction_dir / config.raw_text_filename
out_path = non_instruction_dir / config.non_instruction_filename
raw_text = in_path.read_text(encoding='utf-8')
def clean_text(text: str) -> str:
    text = re.sub(r'\bX{2,}\b', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' [EMAIL] ', text)
    text = re.sub(r'\b\d{5,}\b', ' [NUMBER] ', text)
    text = re.sub(r'[^\w\s\.,!?;:"()\-\/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
paragraphs = [p.strip() for p in raw_text.split('\n\n') if p.strip()]
cleaned = [clean_text(p) for p in paragraphs]
cleaned = [p for p in cleaned if len(p) >= config.min_chars_per_paragraph]
out_path.write_text('\n\n'.join(cleaned), encoding='utf-8')
logger.info('Cleaning complete')
logger.info('Input paragraphs: %s', len(paragraphs))
logger.info('Final non-instruction paragraphs: %s', len(cleaned))
logger.info('Saved final file: %s', out_path.resolve())
print('Final paragraphs:', len(cleaned))
print('Saved to:', out_path.resolve())

In [ ]:
# 10) Create Hugging Face dataset
from datasets import Dataset, DatasetDict
input_path = non_instruction_dir / config.non_instruction_filename
dataset_dir = non_instruction_dir / config.hf_dataset_dirname
text = input_path.read_text(encoding='utf-8')
paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
records = [{'text': p} for p in paragraphs]
full_ds = Dataset.from_list(records)
split_ds = full_ds.train_test_split(test_size=config.test_size, seed=config.seed)
hf_ds = DatasetDict({'train': split_ds['train'], 'validation': split_ds['test']})
dataset_dir.mkdir(parents=True, exist_ok=True)
hf_ds.save_to_disk(str(dataset_dir))
logger.info('HF dataset creation complete')
logger.info('Total rows: %s', len(full_ds))
logger.info('Train rows: %s', len(hf_ds['train']))
logger.info('Validation rows: %s', len(hf_ds['validation']))
print(hf_ds)

In [ ]:
# 11) Load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded:', config.model_name)
print('Vocab size:', len(tokenizer))

In [ ]:
# 12) Tokenization and text packing
dataset = hf_ds
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=config.block_size)
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing text corpus',
)
tokenized_datasets

In [ ]:
# 13) Create fixed-size training blocks
def create_training_blocks(tokenized_examples):
    all_input_ids = []
    all_attention_masks = []
    for input_ids in tokenized_examples['input_ids']:
        all_input_ids.extend(input_ids)
    for attention_mask in tokenized_examples['attention_mask']:
        all_attention_masks.extend(attention_mask)
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size
    if usable_tokens == 0:
        return {'input_ids': [], 'attention_mask': [], 'labels': []}
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]
    input_id_blocks = []
    attention_mask_blocks = []
    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size
        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])
    labels = [block.copy() for block in input_id_blocks]
    return {'input_ids': input_id_blocks, 'attention_mask': attention_mask_blocks, 'labels': labels}
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f'Creating fixed-size training blocks of {config.block_size} tokens',
)
print(final_dataset)
print(final_dataset['train'][0].keys())

In [ ]:
# 14) Check device availability
import torch
use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
if use_cuda:
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 15) Load base model
import gc
from transformers import AutoModelForCausalLM
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
base_model.config.use_cache = False
print('Base model loaded successfully.')

In [ ]:
# 16) Apply LoRA adapters
from peft import LoraConfig, TaskType, get_peft_model
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 17) Train model
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
train_dataset = final_dataset['train']
eval_dataset = final_dataset['validation']

# Keep only one sample so each epoch is one training step (3 epochs => 3 steps).
train_dataset = train_dataset.select(range(min(1, len(train_dataset))))
eval_dataset = eval_dataset.select(range(min(1, len(eval_dataset))))

training_args = TrainingArguments(
    output_dir=str(project_root / config.output_dir),
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=config.weight_decay,
    logging_steps=1,
    logging_first_step=True,
    eval_steps=1,
    save_steps=1,
    save_total_limit=config.save_total_limit,
    max_steps=-1,
    evaluation_strategy='steps',
    save_strategy='steps',
    bf16=use_cuda,
    fp16=False,
    report_to='none',
)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)
train_result = trainer.train()
print('Training completed.')